# 1. 초기 MLP 모델(중간 발표용) PyTorch 모델 입력을 위한 dataclass 부분 gemini 참조


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 하이퍼 파라미터 설정
INPUT_FEATURES = 30
LEARNING_RATE = 0.001
BATCH_SIZE = 1024
EPOCHS = 10

try:
    df = pd.read_csv('scaled.csv')
except FileNotFoundError:
    print("Error: 'creditcard.csv' 파일을 찾을 수 없습니다.")
    exit()
# 1. (확인) 'Class' 열에 NaN이 있는지 확인
nan_count = df['Class'].isnull().sum()
if nan_count > 0:
    print(f"경고: 'Class' 열에 {nan_count}개의 NaN(결측치)이 있습니다. 이 행들을 제거합니다.")
    # 2. (해결) 'Class' 열에 NaN이 있는 행(row)을 제거
    df.dropna(subset=['Class'], inplace=True)

# 3. 이제 NaN이 없는 깨끗한 데이터로 X, y 분리
X_raw = df.drop('Class', axis=1)
y_raw = df['Class']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

input_dim = X_train.shape[1]
print(f"입력 피처 개수: {input_dim}")

# 1. PyTorch 모델 입력을 위한 Dataset 클래스 정의
class CreditCardDataset(Dataset):
    def __init__(self, X, y):
        # 1-1. NumPy 배열을 PyTorch 텐서로 변환
        self.X = torch.tensor(X, dtype=torch.float32)

        # 1-2. (중요) BCEWithLogitsLoss는 (N, 1) 형태의
        #      타겟을 기대하므로 .reshape(-1, 1)이 필수
        self.y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

    def __len__(self):
        # 1-3. 전체 데이터 샘플의 개수를 반환
        return len(self.y)

    def __getitem__(self, idx):
        # 1-4. [idx]번째의 (X, y) 데이터 쌍을 반환
        return self.X[idx], self.y[idx]

# 2. 전처리된 데이터로 Dataset 인스턴스(객체) 생성
train_dataset = CreditCardDataset(X_train, y_train.values)
test_dataset = CreditCardDataset(X_test, y_test.values)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"DataLoader 생성 완료. 훈련 배치의 수: {len(train_loader)}")

# 3. PyTorch 모델 정의 (간단한 MLP)
class FraudDetector(nn.Module):
  def __init__(self,input_features):
    super(FraudDetector,self).__init__()
    self.layer1 = nn.Linear(input_features,64)
    self.layer2 = nn.Linear(64,32)
    self.dropout = nn.Dropout(0.3)
    self.output_layer = nn.Linear(32,1)
    self.relu = nn.ReLU()
  def forward(self,x):
    x = self.relu(self.layer1(x))
    x = self.dropout(x) # 드롭아웃 적용
    x = self.relu(self.layer2(x))
    x = self.dropout(x) # 드롭아웃 적용
    x = self.output_layer(x)
    return x
model = FraudDetector(input_features = input_dim)
print("모델 구조:")
print(model)

# 4. 불균형 처리를 위한 가중치 계산
count_normal = (y_train == 0).sum()
count_fraud = (y_train == 1).sum()

# 이 값을 텐서로 변환 (CPU/GPU 장치에 맞게)
pos_weight_value = count_normal / count_fraud
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

print(f"정상 샘플 수 (Train): {count_normal}")
print(f"사기 샘플 수 (Train): {count_fraud}")
print(f"사기(1) 클래스 가중치 (pos_weight): {pos_weight_tensor.item():.2f}")

# 손실 함수 정의
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# 옵티마이저 정의
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.to(device) # 모델을 GPU/CPU로 보냄

# -------------------------------------------------------------------
# 5. 훈련 (Training) 루프
# -------------------------------------------------------------------
print("\n모델 훈련 시작...")
for epoch in range(EPOCHS):
    model.train() # 훈련 모드 (Dropout 활성화)
    total_loss = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Forward pass (예측)
        outputs = model(inputs)

        # 2. Loss 계산 (가중치 적용된 손실)
        loss = criterion(outputs, labels)

        # 3. Backward pass and optimization (역전파 및 가중치 업데이트)
        optimizer.zero_grad() # 그래디언트 초기화
        loss.backward()       # 역전파
        optimizer.step()      # 가중치 업데이트

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}')

print("훈련 완료.")

# -------------------------------------------------------------------
# 6. 평가 (Evaluation) 루프
# -------------------------------------------------------------------
print("\n모델 평가 시작...")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad(): # 그래디언트 계산 비활성화 (메모리 절약, 속도 향상)
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)

        # 출력(Logits)을 확률(Sigmoid)로 변환
        probs = torch.sigmoid(outputs)

        # 0.5 임계값을 기준으로 0 또는 1로 변환
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n=== 최종 평가 리포트 ===")
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=['Class 0 (Normal)', 'Class 1 (Fraud)'], digits=4))

입력 피처 개수: 30
DataLoader 생성 완료. 훈련 배치의 수: 223
모델 구조:
FraudDetector(
  (layer1): Linear(in_features=30, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)
정상 샘플 수 (Train): 227451
사기 샘플 수 (Train): 394
사기(1) 클래스 가중치 (pos_weight): 577.29

모델 훈련 시작...
Epoch [1/10], Loss: 0.7127
Epoch [2/10], Loss: 0.3767
Epoch [3/10], Loss: 0.3104
Epoch [4/10], Loss: 0.2722
Epoch [5/10], Loss: 0.2754
Epoch [6/10], Loss: 0.2599
Epoch [7/10], Loss: 0.2330
Epoch [8/10], Loss: 0.2353
Epoch [9/10], Loss: 0.2219
Epoch [10/10], Loss: 0.1999
훈련 완료.

모델 평가 시작...

=== 최종 평가 리포트 ===
[[56181   683]
 [   10    88]]
                  precision    recall  f1-score   support

Class 0 (Normal)     0.9998    0.9880    0.9939     56864
 Class 1 (Fraud)     0.1141    0.8980    0.2025        98

        accuracy                         0.9878     56962
       ma

- 간단한 nn.Linear와 ReLU를 사용한 MLP
- 데이터 불균형을 고려하기 위해, nn.BCEWithLogitsLoss의 pos_weight를 사용해 손실 함수에서 가중치를 조절 => 사기를 정상으로 판정한 경우, 파라미터 수정을 강제하기 위한 장치
- 결과적으로 사기건에 대해, 90%의 정확성을 보임 그러나, 사기 건을 오판하지 않기 위해 정상건 1682건도 사기로 탐지 => 사기 건 탐지에 있어서 정확도는 낮게 나오게 됨
- 정상건에 대한 사기 탐지를 줄이기 위해, sigmoid 기준 값을 조정

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 하이퍼 파라미터 설정
INPUT_FEATURES = 30
LEARNING_RATE = 0.001
BATCH_SIZE = 1024
EPOCHS = 10

try:
    df = pd.read_csv('scaled.csv')
except FileNotFoundError:
    print("Error: 'creditcard.csv' 파일을 찾을 수 없습니다.")
    exit()
# 1. (확인) 'Class' 열에 NaN이 있는지 확인
nan_count = df['Class'].isnull().sum()
if nan_count > 0:
    print(f"경고: 'Class' 열에 {nan_count}개의 NaN(결측치)이 있습니다. 이 행들을 제거합니다.")
    # 2. (해결) 'Class' 열에 NaN이 있는 행(row)을 제거
    df.dropna(subset=['Class'], inplace=True)

# 3. 이제 NaN이 없는 깨끗한 데이터로 X, y 분리
X_raw = df.drop('Class', axis=1)
y_raw = df['Class']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

input_dim = X_train.shape[1] # 아마도 29 (V1-V28, Amount)
print(f"입력 피처 개수: {input_dim}")

# 1. PyTorch 모델 입력을 위한 Dataset 클래스 정의
class CreditCardDataset(Dataset):
    def __init__(self, X, y):
        # 1-1. NumPy 배열을 PyTorch 텐서로 변환
        self.X = torch.tensor(X, dtype=torch.float32)

        # 1-2. (중요) BCEWithLogitsLoss는 (N, 1) 형태의
        #      타겟을 기대하므로 .reshape(-1, 1)이 필수
        self.y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

    def __len__(self):
        # 1-3. 전체 데이터 샘플의 개수를 반환
        return len(self.y)

    def __getitem__(self, idx):
        # 1-4. [idx]번째의 (X, y) 데이터 쌍을 반환
        return self.X[idx], self.y[idx]

# 2. 전처리된 데이터로 Dataset 인스턴스(객체) 생성
train_dataset = CreditCardDataset(X_train, y_train.values)
test_dataset = CreditCardDataset(X_test, y_test.values)

# 3. PyTorch 모델 정의 (간단한 MLP)
class FraudDetector(nn.Module):
  def __init__(self,input_features):
    super(FraudDetector,self).__init__()
    self.layer1 = nn.Linear(input_features,64)
    self.layer2 = nn.Linear(64,32)
    self.dropout = nn.Dropout(0.3)
    self.output_layer = nn.Linear(32,1)
    self.relu = nn.ReLU()
  def forward(self,x):
    x = self.relu(self.layer1(x))
    x = self.dropout(x) # 드롭아웃 적용
    x = self.relu(self.layer2(x))
    x = self.dropout(x) # 드롭아웃 적용
    # 출력층에 Sigmoid 없음! -> BCEWithLogitsLoss가 처리
    x = self.output_layer(x)
    return x
model = FraudDetector(input_features = input_dim)
print("모델 구조:")
print(model)
# 4. 불균형 처리를 위한 가중치 계산
count_normal = (y_train == 0).sum()
count_fraud = (y_train == 1).sum()

# 이 값을 텐서로 변환 (CPU/GPU 장치에 맞게)
pos_weight_value = count_normal / count_fraud
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

print(f"정상 샘플 수 (Train): {count_normal}")
print(f"사기 샘플 수 (Train): {count_fraud}")
print(f"사기(1) 클래스 가중치 (pos_weight): {pos_weight_tensor.item():.2f}")

# 손실 함수 정의
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# 옵티마이저 정의
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.to(device) # 모델을 GPU/CPU로 보냄

# -------------------------------------------------------------------
# 5. 훈련 (Training) 루프
# -------------------------------------------------------------------
print("\n모델 훈련 시작...")
for epoch in range(EPOCHS):
    model.train() # 훈련 모드 (Dropout 활성화)
    total_loss = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Forward pass (예측)
        outputs = model(inputs)

        # 2. Loss 계산 (가중치 적용된 손실)
        loss = criterion(outputs, labels)

        # 3. Backward pass and optimization (역전파 및 가중치 업데이트)
        optimizer.zero_grad() # 그래디언트 초기화
        loss.backward()       # 역전파
        optimizer.step()      # 가중치 업데이트

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}')

print("훈련 완료.")

# -------------------------------------------------------------------
# 6. 평가 (Evaluation) 루프
# -------------------------------------------------------------------
print("\n모델 평가 시작...")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad(): # 그래디언트 계산 비활성화 (메모리 절약, 속도 향상)
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)

        # 출력(Logits)을 확률(Sigmoid)로 변환
        probs = torch.sigmoid(outputs)
        THRESHOLD = 0.8
        preds = (probs > THRESHOLD).float()


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n=== 최종 평가 리포트 ===")
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=['Class 0 (Normal)', 'Class 1 (Fraud)'], digits=4))

입력 피처 개수: 30
모델 구조:
FraudDetector(
  (layer1): Linear(in_features=30, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)
정상 샘플 수 (Train): 227451
사기 샘플 수 (Train): 394
사기(1) 클래스 가중치 (pos_weight): 577.29

모델 훈련 시작...
Epoch [1/10], Loss: 0.7040
Epoch [2/10], Loss: 0.3535
Epoch [3/10], Loss: 0.2928
Epoch [4/10], Loss: 0.2829
Epoch [5/10], Loss: 0.2621
Epoch [6/10], Loss: 0.2307
Epoch [7/10], Loss: 0.2240
Epoch [8/10], Loss: 0.1906
Epoch [9/10], Loss: 0.2021
Epoch [10/10], Loss: 0.1905
훈련 완료.

모델 평가 시작...

=== 최종 평가 리포트 ===
[[56639   225]
 [   12    86]]
                  precision    recall  f1-score   support

Class 0 (Normal)     0.9998    0.9960    0.9979     56864
 Class 1 (Fraud)     0.2765    0.8776    0.4205        98

        accuracy                         0.9958     56962
       macro avg     0.6382    0.9368    